# **Détection des dattes fruits avec YOLOv8**

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import random





* ***Préparer Data pour entrainement avec Yolov8*** : (labels + images)

In [ ]:
BASE_DIR = "../data/dataset_detection/"

# IMAGES_DIR = f"{BASE_DIR}/data_merged/images"
# LABELS_DIR = f"{BASE_DIR}/data_merged/labels"




IMAGES_DIR = os.path.join(BASE_DIR, "data_merged/images")
LABELS_DIR = os.path.join(BASE_DIR, "data_merged/labels")


IMAGE_EXT = [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]

In [ ]:
print("Images :", len(os.listdir(IMAGES_DIR)))
print("Labels :", len(os.listdir(LABELS_DIR)))

*  vérifie chaque image

*  affiche celles qui n'ont pas de label

*  les supprime





In [ ]:


def remove_images_without_labels(images_dir, labels_dir, image_ext, show_images=True):
    """
    Vérifie les images sans label YOLO et les supprime.

    Args:
        images_dir : dossier des images
        labels_dir : dossier des annotations
        image_ext : extensions autorisées
        show_images : afficher les images supprimées
    """

    missing_images = []

    # vérifier les labels
    for img in os.listdir(images_dir):

        if os.path.splitext(img)[1].lower() in image_ext:

            name = os.path.splitext(img)[0]
            label_path = os.path.join(labels_dir, name + ".txt")

            if not os.path.exists(label_path):
                missing_images.append(img)

    print("Images sans label :", len(missing_images))

    # afficher les images
    if show_images:
        for img in missing_images:

            img_path = os.path.join(images_dir, img)

            image = cv2.imread(img_path)

            if image is None:
                continue

            # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # plt.figure(figsize=(4,4))
            # plt.imshow(image)
            # plt.title(f"Sans label : {img}")
            # plt.axis("off")
            # plt.show()

    # supprimer les images
    for img in missing_images:
        os.remove(os.path.join(images_dir, img))

    print(f"🗑️ {len(missing_images)} images supprimées")

    return missing_images

In [ ]:
remove_images_without_labels(
    IMAGES_DIR,
    LABELS_DIR,
    IMAGE_EXT
)


In [ ]:
print("Images :", len(os.listdir(IMAGES_DIR)))
print("Labels :", len(os.listdir(LABELS_DIR)))

***

* Afficher une échantillon des images :

In [ ]:



# -----------------------------
# lire les annotations YOLO
# -----------------------------
def read_yolo_label(label_path):

    boxes = []

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        cls, xc, yc, w, h = map(float, line.split())
        boxes.append((cls, xc, yc, w, h))

    return boxes


# -----------------------------
# dessiner bounding boxes
# -----------------------------
def draw_boxes(image, boxes):

    h, w, _ = image.shape

    for cls, xc, yc, bw, bh in boxes:

        x_center = xc * w
        y_center = yc * h

        box_w = bw * w
        box_h = bh * h

        x_min = int(x_center - box_w / 2)
        y_min = int(y_center - box_h / 2)

        x_max = int(x_center + box_w / 2)
        y_max = int(y_center + box_h / 2)

        cv2.rectangle(image, (x_min, y_min), (x_max, y_max), (0,0,255), 6)

    return image


# -----------------------------
# afficher échantillons
# -----------------------------
def show_random_samples(n_samples=9):

    images = os.listdir(IMAGES_DIR)
    samples = random.sample(images, n_samples)

    cols = 3
    rows = (n_samples // cols) + 1

    plt.figure(figsize=(12,12))

    for i, img_name in enumerate(samples):

        img_path = os.path.join(IMAGES_DIR, img_name)
        label_path = os.path.join(LABELS_DIR, os.path.splitext(img_name)[0] + ".txt")

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if os.path.exists(label_path):
            boxes = read_yolo_label(label_path)
            img = draw_boxes(img, boxes)

        plt.subplot(rows, cols, i+1)
        plt.imshow(img)
        plt.title(img_name)
        plt.axis("off")

    plt.show()


# -----------------------------
# exécuter
# -----------------------------
show_random_samples(9)

## Split Data to Train, Test, Val (70%, 20%, 10%)

1️⃣ mélanger les images
2️⃣ faire le split 70 / 20 / 10
3️⃣ déplacer image + label correspondant ensemble
4️⃣ créer la structure correcte.

In [ ]:


def split_detection_dataset(images_dir, labels_dir, base_dir, image_ext,
                            train_ratio=0.7, val_ratio=0.2, test_ratio=0.1):
    """
    Split un dataset de détection en train / val / test (format YOLO)
    """

    # assert train_ratio + val_ratio + test_ratio == 1.0, "Les ratios doivent faire 1"

    # créer dossiers
    splits = ["data_splited/train", "data_splited/val", "data_splited/test"]

    for split in splits:
        os.makedirs(os.path.join(base_dir, split, "images"), exist_ok=True)
        os.makedirs(os.path.join(base_dir, split, "labels"), exist_ok=True)

    # récupérer images
    images = [f for f in os.listdir(images_dir) if f.lower().endswith(tuple(image_ext))]

    random.shuffle(images)

    n_total = len(images)
    n_train = int(n_total * train_ratio)
    n_val = int(n_total * val_ratio)

    train_images = images[:n_train]
    val_images = images[n_train:n_train+n_val]
    test_images = images[n_train+n_val:]


    def move_files(image_list, split):

        for img in image_list:

            name = os.path.splitext(img)[0]

            img_src = os.path.join(images_dir, img)
            lbl_src = os.path.join(labels_dir, name + ".txt")

            img_dst = os.path.join(base_dir, split, "images", img)
            lbl_dst = os.path.join(base_dir, split, "labels", name + ".txt")

            shutil.move(img_src, img_dst)

            if os.path.exists(lbl_src):
                shutil.move(lbl_src, lbl_dst)


    # déplacer fichiers
    move_files(train_images, "data_splited/train")
    move_files(val_images, "data_splited/val")
    move_files(test_images, "data_splited/test")

    print("\n✅ Dataset split terminé")
    print("Train :", len(train_images))
    print("Val :", len(val_images))
    print("Test :", len(test_images))

In [ ]:
# IMAGES_DIR = os.path.join(BASE_DIR, "data_splited/images")
# LABELS_DIR = os.path.join(BASE_DIR, "data_splited/labels")

train_ratio=0.7
val_ratio=0.2
test_ratio = 1 - train_ratio - val_ratio

split_detection_dataset(
    IMAGES_DIR,
    LABELS_DIR,
    BASE_DIR,
    IMAGE_EXT,
    train_ratio=train_ratio,
    val_ratio=val_ratio, 
    test_ratio=test_ratio
)

In [ ]:
import os
import matplotlib.pyplot as plt

BASE_DIR = "../data/dataset_detection/data_splited/"

def count_images():

    train_count = len(os.listdir(os.path.join(BASE_DIR, "train/images")))
    val_count = len(os.listdir(os.path.join(BASE_DIR, "val/images")))
    test_count = len(os.listdir(os.path.join(BASE_DIR, "test/images")))

    return {
        "train": train_count,
        "val": val_count,
        "test": test_count
    }


def plot_distribution(data):

    labels = list(data.keys())
    values = list(data.values())

    plt.figure(figsize=(6,4))
    plt.bar(labels, values)

    plt.title("Dataset Split Distribution")
    plt.xlabel("Dataset Split")
    plt.ylabel("Number of Images")

    for i,v in enumerate(values):
        plt.text(i, v+5, str(v), ha='center')

    plt.show()


data = count_images()
plot_distribution(data)

In [ ]:

list_dirs = {
    "All": "../data/dataset_detection/data_merged",
    "Train": "../data/dataset_detection/data_splited/train",
    "Val": "../data/dataset_detection/data_splited/val",
    "Test": "../data/dataset_detection/data_splited/test",
}


def count_data_splited_part(list_dir):

    for name, path in list_dir.items():

        images_dir = os.path.join(path, "images")
        labels_dir = os.path.join(path, "labels")

        n_images = len(os.listdir(images_dir)) if os.path.exists(images_dir) else 0
        n_labels = len(os.listdir(labels_dir)) if os.path.exists(labels_dir) else 0

        print(f"📂 {name}")
        print(f"Images : {n_images}")
        print(f"Labels : {n_labels}")
        print("=" * 40)


count_data_splited_part(list_dirs)

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="../data/dataset_detection/data_splited/data.yaml",
    imgsz=640,
    epochs=1,
    batch=16
)

In [ ]:
from ultralytics import YOLO

def evaluate_model_detection(model_path, data_yaml, split="test", imgsz=640, batch=8, device="cpu"):
    """
    Évaluer un modèle YOLOv8 sur un dataset.

    Args:
        model_path : chemin vers best.pt
        data_yaml : chemin vers data.yaml
        split : train / val / test
        imgsz : taille des images
        batch : batch size
        device : cpu ou cuda

    Returns:
        dict contenant les métriques
    """

    model = YOLO(model_path)

    metrics = model.val(
        data=data_yaml,
        split=split,
        imgsz=imgsz,
        batch=batch,
        device=device,
        verbose=False
    )

    results = {
        "mAP50-95": metrics.box.map,
        "mAP50": metrics.box.map50,
        "precision": metrics.box.p,
        "recall": metrics.box.r
    }

    print("\n📊 Résultats de l'évaluation")
    print("-" * 40)
    print(f"mAP50-95 : {results['mAP50-95']:.4f}")
    print(f"mAP50    : {results['mAP50']:.4f}")
    print(f"Precision: {results['precision']:.4f}")
    print(f"Recall   : {results['recall']:.4f}")

    return results

In [ ]:
results = evaluate_model_detection(
    model_path="../analyse_configuration/detection/runs/detect/train/weights/best.pt",
    data_yaml="../data/dataset_detection/data_splited/data.yaml",
    split="val",
    imgsz=640,
    batch=9,
    device="cpu"
)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

RESULTS_DIR = "./runs/detect/val4"

images = [
    "confusion_matrix",
    'BoxF1_curve',
    'BoxP_curve',
    'BoxPR_curve',
    'BoxR_curve'
]

def show_evaluation_plots():

    plt.figure(figsize=(16,20))

    for i, img_name in enumerate(images):

        path = os.path.join(RESULTS_DIR, img_name + ".png")

        if os.path.exists(path):

            img = cv2.imread(path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            plt.subplot(3,2,i+1)
            plt.imshow(img)
            plt.title(img_name)
            plt.axis("off")

    plt.tight_layout()
    plt.show()


show_evaluation_plots()

In [ ]:
import os
import random
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO


def predict_random_image_detection(model, image_dir, conf=0.25):
    """
    Prédire sur une image aléatoire d'un dossier avec YOLOv8.

    Args:
        model : modèle YOLO chargé
        image_dir : dossier contenant les images
        conf : seuil de confiance
    """

    images_list = os.listdir(image_dir)

    if len(images_list) == 0:
        print("❌ Aucun fichier image trouvé")
        return

    random_image = random.choice(images_list)

    image_path = os.path.join(image_dir, random_image)

    print(f"📷 Image choisie : {random_image}")

    predictions = model.predict(
        source=image_path,
        conf=conf
    )

    annotated_img = predictions[0].plot()

    annotated_img = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(8,8))
    plt.imshow(annotated_img)
    plt.axis("off")
    plt.show()

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train/weights/best.pt")

predict_random_image_detection(
    model,
    "../data/dataset_detection/data_splited/test/images"
)

* Sauvegarder le models : 

In [ ]:
model.save("../models/date_detector_model.pt")